# EXP-01: Mean Pooling + Linear

**Model:** `mean_linear` — Mean Pool CLIP features + Linear classifier  
**Params:** ~5.6M | **Batch:** 32 | **Epochs:** 10  
**W&B project:** `blip2-vqa-experiment`

Thu tu chay:
1. Runtime > Change runtime type > **T4 GPU**
2. Chay Cell 1 (Mount Drive + Pull repo)
3. Chay Cell 2 (Cai deps + W&B login)
4. Sua `YOUR_NAME` o Cell 3 roi chay
5. Chay Cell 4 (Pre-extract — bo qua neu da co cache)
6. Chay Cell 5 (Train)
7. Chay Cell 6 (Evaluate)
8. Neu Colab disconnect: Cell 1 + 2 + 3 + Cell 7 (Resume)

## Cell 1 — Mount Drive & Pull repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = "/content/blip2-fusion-experiment-vqa"
GITHUB_USER = "<username>"  # <-- SUA THANH USERNAME GITHUB CUA BAN

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/blip2-fusion-experiment-vqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

## Cell 2 — Cai dependencies & Dang nhap W&B

In [ ]:
!pip install -r requirements.txt -q

import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

import wandb
wandb.login()  # nhap API key tu wandb.ai/authorize

## Cell 3 — Khai bao bien

**Chi can sua `YOUR_NAME`** — cac duong dan da duoc dinh nghia trong `configs/exp01.yaml`.

In [ ]:
YOUR_NAME = "ten"   # <-- SUA: ten viet tat (vd: khoa, minh, tuan)

# Bien duoc tao tu dong — KHONG can sua
CONFIG     = "configs/exp01.yaml"
RUN_NAME   = f"exp01_lan1_{YOUR_NAME}"

# Doc tu YAML de hien thi thong tin
import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

CHECKPOINT = cfg["training"]["output_dir"] + "/best_model.pth"
EVAL_OUT   = cfg["training"]["output_dir"] + "/eval_results.json"

print(f"Config      : {CONFIG}")
print(f"Run name    : {RUN_NAME}")
print(f"Data root   : {cfg['data']['data_root']}")
print(f"Output dir  : {cfg['training']['output_dir']}")
print(f"Epochs      : {cfg['training']['num_epochs']}")
print(f"Batch size  : {cfg['data']['batch_size']}")
print(f"LR          : {cfg['training']['learning_rate']}")

## Cell 4 — Pre-extract CLIP features

**Bo qua cell nay neu da co cache.** Chi chay 1 lan dau (~15-20 phut tren T4).

In [ ]:
import os, h5py

data_root = cfg['data']['data_root']
cache_dir = os.path.join(data_root, cfg['data']['cache_dir'])
train_h5  = os.path.join(cache_dir, "train_features.h5")
val_h5    = os.path.join(cache_dir, "val_features.h5")

if os.path.exists(train_h5) and os.path.exists(val_h5):
    with h5py.File(train_h5, "r") as f:
        print(f"train_features.h5 : {len(f.keys()):,} anh")
    with h5py.File(val_h5, "r") as f:
        print(f"val_features.h5   : {len(f.keys()):,} anh")
    print("Cache da co — bo qua pre-extract.")
else:
    vqav2_dir = cfg['data']['vqav2_dir']
    print("Bat dau pre-extract...")
    !python data/pre_extract_features.py \
        --split      both \
        --data_root  "{data_root}" \
        --output_dir "{cache_dir}" \
        --vqav2_dir  "{vqav2_dir}" \
        --batch_size 64 \
        --ckpt_every 10
    print("Pre-extract hoan thanh.")

## Cell 5 — Train

In [ ]:
!python scripts/train.py \
    --config   {CONFIG} \
    --run_name {RUN_NAME}

## Cell 6 — Evaluate

In [ ]:
!python scripts/evaluate.py \
    --config     {CONFIG} \
    --checkpoint {CHECKPOINT} \
    --split      val \
    --output     {EVAL_OUT}

# In ket qua
import json
if os.path.exists(EVAL_OUT):
    with open(EVAL_OUT) as f:
        r = json.load(f)
    print("\n" + "=" * 45)
    print(f"KET QUA — {RUN_NAME}")
    print("=" * 45)
    for k, v in r.items():
        if isinstance(v, float):
            print(f"  {k:<20}: {v*100:.2f}%" if "acc" in k.lower() or k in ("overall","yes/no","number","other") else f"  {k:<20}: {v:.4f}")
        else:
            print(f"  {k:<20}: {v}")
    print("=" * 45)
    print(f"\nGhi vao bang theo doi:")
    print(f"  Run name : {RUN_NAME}")
    print(f"  Val Acc  : {r.get('overall', r.get('val_acc', '?'))}")

## Cell 7 — Resume (sau khi Colab disconnect)

Chay **Cell 1 + 2 + 3** truoc, sau do chay cell nay.  
Giu nguyen `YOUR_NAME` giong lan dau de W&B tiep tuc cung 1 run.

In [ ]:
!python scripts/train.py \
    --config   {CONFIG} \
    --run_name {RUN_NAME} \
    --resume   auto